# 🧠 NEXUS Stock AI — Phases 7, 8, & 9
## High-Speed FinBERT GPU Batch Sentiment Analysis

**Objective:** Execute high-throughput batched sentiment classification on ~62,000 financial headlines from `aligned_news.parquet` using the **ProsusAI/finbert** model on the **Colab T4 GPU**.

### ⚡ Key Optimizations & Pipeline Components
1. **Phase 7 (Load FinBERT on GPU):** Initialize `ProsusAI/finbert` via `transformers.pipeline` targeting `device=0` (NVIDIA T4) with `top_k=None` to capture full 3-class probability distributions (`positive`, `neutral`, `negative`).
2. **Phase 8 (High-Speed Batch Inference):** Batch headlines with `batch_size=256` and short sequence length (`max_length=64`) under `torch.inference_mode()` to saturate GPU Tensor Cores without OOM.
3. **Phase 9 (Sentiment Scoring & Persistence):** Compute net sentiment score:
   $$\text{sentiment\_score} = \text{positive\_prob} - \text{negative\_prob} \in [-1.0, +1.0]$$
   Persist to Google Drive (`sentiment_news.parquet`), refresh `nexus_data_backup.zip` for instant offline download, and clear CUDA cache.

### ⚙️ Step 1: Environment Setup & Google Drive Mount

In [1]:
# Install required NLP libraries if not present
%pip install -q transformers torch pyarrow tqdm

import os
import sys
import gc
import time
import shutil
import zipfile
import torch
import pandas as pd
import numpy as np
from transformers import pipeline
from tqdm.auto import tqdm

# Mount Google Drive
try:
    from google.colab import drive
    print("Mounting Google Drive at /content/drive...")
    drive.mount("/content/drive")
    print("✓ Google Drive mounted successfully.")
except Exception as e:
    print(f"Drive mount note: {e}")

# Check GPU
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"✓ GPU Active: {gpu_name} ({gpu_vram:.2f} GB VRAM)")
    device_id = 0
else:
    print("⚠️ No GPU detected. Running on CPU (slower).")
    device_id = -1

GDRIVE_DIR = "/content/drive/MyDrive/NEXUS_Stock_AI/data"
LOCAL_DIR = "./data"
INPUT_PARQUET = os.path.join(GDRIVE_DIR, "aligned_news.parquet") if os.path.exists(os.path.join(GDRIVE_DIR, "aligned_news.parquet")) else os.path.join(LOCAL_DIR, "aligned_news.parquet")
print(f"Target Input: {INPUT_PARQUET} (Exists: {os.path.exists(INPUT_PARQUET)})")

Mounting Google Drive at /content/drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ Google Drive mounted successfully.
✓ GPU Active: Tesla T4 (14.56 GB VRAM)
Target Input: /content/drive/MyDrive/NEXUS_Stock_AI/data/aligned_news.parquet (Exists: True)


### 📥 Step 2: Load Aligned News & Prepare Target Headlines
Financial headlines (`Article_title`) contain dense market sentiment and zero missing values.

In [2]:
print("Loading aligned news dataset...")
news_df = pd.read_parquet(INPUT_PARQUET)
print(f"Loaded {len(news_df):,} rows from {INPUT_PARQUET}")

# Cast Article_title to clean python list of strings
titles = news_df["Article_title"].fillna("").astype(str).tolist()
print(f"Prepared {len(titles):,} headline strings for inference.")
print(f"Sample headline: '{titles[0]}'")

Loading aligned news dataset...
Loaded 61,992 rows from /content/drive/MyDrive/NEXUS_Stock_AI/data/aligned_news.parquet
Prepared 61,992 headline strings for inference.
Sample headline: 'Nvidia Goes Negative (NVDA)'


### 🤖 Step 3: Phase 7 — Initialize FinBERT on GPU
- Model: `ProsusAI/finbert` (pre-trained specifically on financial text: Corporate filings, earnings calls, and news).
- `device=0`: Pin pipeline directly to the Colab T4 GPU.
- `top_k=None`: Ensures the pipeline returns probabilities for all 3 classes (`positive`, `neutral`, `negative`).

In [3]:
print("Initializing FinBERT text-classification pipeline...")
start_model = time.time()

pipe = pipeline(
    "text-classification",
    model="ProsusAI/finbert",
    device=device_id,
    top_k=None,  # Returns exact probabilities for positive, neutral, and negative
    truncation=True,
    max_length=64
)

print(f"✓ FinBERT loaded in {time.time() - start_model:.2f}s")

Initializing FinBERT text-classification pipeline...


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  438MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

...

model.safetensors: downloading bytes:           |  0.00B            

tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

✓ FinBERT loaded in 17.40s


### ⚡ Step 4: Phase 8 — High-Speed Batched Inference
- `batch_size = 256`: Fully saturates T4 GPU memory bandwidth.
- `max_length = 64`: Optimized for concise headlines, eliminating redundant padding tokens.
- `torch.inference_mode()`: Bypasses autograd tracking for maximum inference speed.
- Output parsing maps dictionary scores to `positive_prob`, `neutral_prob`, and `negative_prob`.

In [4]:
BATCH_SIZE = 256
MAX_LENGTH = 64

print(f"Starting batch inference on {len(titles):,} headlines (batch_size={BATCH_SIZE})...")
start_infer = time.time()

pos_probs = []
neu_probs = []
neg_probs = []

with torch.inference_mode():
    for i in tqdm(range(0, len(titles), BATCH_SIZE), desc="FinBERT Inference", unit="batch"):
        batch_titles = titles[i : i + BATCH_SIZE]
        batch_results = pipe(
            batch_titles,
            batch_size=len(batch_titles),
            truncation=True,
            max_length=MAX_LENGTH
        )
        
        # Parse nested dict output
        for item in batch_results:
            score_map = {entry["label"].lower(): float(entry["score"]) for entry in item}
            pos_probs.append(score_map.get("positive", 0.0))
            neu_probs.append(score_map.get("neutral", 0.0))
            neg_probs.append(score_map.get("negative", 0.0))

infer_elapsed = time.time() - start_infer
throughput = len(titles) / max(infer_elapsed, 0.001)
print(f"\n✓ Batch inference finished in {infer_elapsed:.2f}s ({infer_elapsed/60:.2f} mins)")
print(f"  • Throughput: {throughput:.1f} headlines/second")

# Assign probabilities to dataframe
news_df["positive_prob"] = np.array(pos_probs, dtype=np.float32)
news_df["neutral_prob"] = np.array(neu_probs, dtype=np.float32)
news_df["negative_prob"] = np.array(neg_probs, dtype=np.float32)

# Free raw probability lists
del titles, pos_probs, neu_probs, neg_probs
if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()

Starting batch inference on 61,992 headlines (batch_size=256)...


FinBERT Inference:   0%|          | 0/243 [00:00<?, ?batch/s]

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset



✓ Batch inference finished in 177.63s (2.96 mins)
  • Throughput: 349.0 headlines/second


220

### 📊 Step 5: Phase 9 — Sentiment Scoring, Persistence, & ZIP Backup
- Net Sentiment Score: `sentiment_score = positive_prob - negative_prob`
- Persist to Google Drive: `/content/drive/MyDrive/NEXUS_Stock_AI/data/sentiment_news.parquet`
- Update `nexus_data_backup.zip` in `/content/` for easy VS Code download.
- Visual validation of 5-row preview and sentiment distributions.

In [5]:
# 1. Calculate net sentiment score
news_df["sentiment_score"] = news_df["positive_prob"] - news_df["negative_prob"]

# 2. Save enriched dataset to Google Drive & Local fallback
target_dir = GDRIVE_DIR if os.path.exists(GDRIVE_DIR) else LOCAL_DIR
os.makedirs(target_dir, exist_ok=True)
os.makedirs(LOCAL_DIR, exist_ok=True)

out_parquet = os.path.join(target_dir, "sentiment_news.parquet")
news_df.to_parquet(out_parquet, engine="pyarrow", compression="zstd", index=False)

local_parquet = os.path.join(LOCAL_DIR, "sentiment_news.parquet")
if os.path.abspath(out_parquet) != os.path.abspath(local_parquet):
    shutil.copy2(out_parquet, local_parquet)

print(f"✓ Saved to Google Drive: {out_parquet} ({os.path.getsize(out_parquet)/(1024*1024):.2f} MB)")
print(f"✓ Local copy synced:    {local_parquet}")

# 3. Update local ZIP backup archive
target_zip = "/content/nexus_data_backup.zip" if os.path.exists("/content") else "./nexus_data_backup.zip"
print(f"\nUpdating local ZIP archive: {target_zip}...")

with zipfile.ZipFile(target_zip, "w", zipfile.ZIP_DEFLATED) as zipf:
    for root, _, files in os.walk(LOCAL_DIR):
        for file in files:
            if file.endswith(".parquet") or file.endswith(".csv"):
                file_path = os.path.join(root, file)
                arcname = os.path.join("data", os.path.relpath(file_path, LOCAL_DIR))
                zipf.write(file_path, arcname)
                print(f"  + Added: {arcname} ({os.path.getsize(file_path)/(1024*1024):.2f} MB)")

zip_size = os.path.getsize(target_zip) / (1024 * 1024)
print(f"✓ Backup ZIP ready: {target_zip} ({zip_size:.2f} MB)")

# 4. Clear CUDA cache
if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()
print("✓ PyTorch CUDA cache cleared successfully.")

✓ Saved to Google Drive: /content/drive/MyDrive/NEXUS_Stock_AI/data/sentiment_news.parquet (92.95 MB)
✓ Local copy synced:    ./data/sentiment_news.parquet

Updating local ZIP archive: /content/nexus_data_backup.zip...
  + Added: data/sentiment_news.parquet (92.95 MB)
  + Added: data/filtered_news.parquet (78.43 MB)
  + Added: data/aligned_news.parquet (91.84 MB)
  + Added: data/cleaned_prices.parquet (1.79 MB)
  + Added: data/filtered_prices.parquet (1.74 MB)
✓ Backup ZIP ready: /content/nexus_data_backup.zip (265.95 MB)
✓ PyTorch CUDA cache cleared successfully.


### 🎯 Step 6: Validation Preview & Summary Statistics

In [6]:
print("=" * 90)
print("🎯 FINBERT SENTIMENT ANALYSIS VALIDATION PREVIEW (5-Row Sample)")
print("=" * 90)

preview_cols = ["Stock_symbol", "Article_title", "sentiment_score", "positive_prob", "negative_prob", "neutral_prob"]
sample_view = news_df[preview_cols].head(5).copy()
display(sample_view)

print("\nSentiment Score Summary Statistics:")
stats_df = pd.DataFrame({
    "Metric": ["Total Rows", "Mean Score", "Median Score", "Std Dev", "Min Score", "Max Score"],
    "Value": [
        f"{len(news_df):,}",
        f"{news_df['sentiment_score'].mean():+.4f}",
        f"{news_df['sentiment_score'].median():+.4f}",
        f"{news_df['sentiment_score'].std():.4f}",
        f"{news_df['sentiment_score'].min():+.4f}",
        f"{news_df['sentiment_score'].max():+.4f}"
    ]
})
display(stats_df)

pos_count = (news_df['sentiment_score'] > 0.05).sum()
neu_count = (news_df['sentiment_score'].abs() <= 0.05).sum()
neg_count = (news_df['sentiment_score'] < -0.05).sum()

print(f"Market Sentiment Breakdown:")
print(f"  • Bullish / Positive (> +0.05):     {pos_count:,d} ({pos_count/len(news_df)*100:.1f}%)")
print(f"  • Neutral ([-0.05, +0.05]):          {neu_count:,d} ({neu_count/len(news_df)*100:.1f}%)")
print(f"  • Bearish / Negative (< -0.05):     {neg_count:,d} ({neg_count/len(news_df)*100:.1f}%)")
print("=" * 90)
print("🎉 PHASES 7, 8, & 9 COMPLETE & VERIFIED!")

🎯 FINBERT SENTIMENT ANALYSIS VALIDATION PREVIEW (5-Row Sample)


,Stock_symbol,Article_title,sentiment_score,positive_prob,negative_prob,neutral_prob
0,NVDA,Nvidia Goes Negative (NVDA),-0.810006,0.015090,0.825096,0.159813
1,NVDA,Auriga Still Not Sure Where Reality Lies For N...,0.061073,0.093324,0.032250,0.874426
2,NVDA,Goldman Sachs Gives Color On Semiconductors (N...,0.017024,0.036286,0.019262,0.944452
3,NVDA,"JP Morgan Upgrades NVIDIA To Neutral, $21 PT",0.305289,0.554447,0.249158,0.196395
4,NVDA,J.P. Morgan Upgrades NVIDIA Corporation To Neu...,0.573761,0.642552,0.068791,0.288657



Sentiment Score Summary Statistics:


,Metric,Value
0,Total Rows,"61,992"
1,Mean Score,+0.0300
2,Median Score,+0.0382
3,Std Dev,0.4958
4,Min Score,-0.9688
5,Max Score,+0.9454


Market Sentiment Breakdown:
  • Bullish / Positive (> +0.05):     28,526 (46.0%)
  • Neutral ([-0.05, +0.05]):          18,021 (29.1%)
  • Bearish / Negative (< -0.05):     15,445 (24.9%)
🎉 PHASES 7, 8, & 9 COMPLETE & VERIFIED!
